[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs/Lab_2_Lagrange_Method.ipynb)


In [ ]:
# 🚀 GOOGLE COLAB SETUP
# If you are running this notebook on Google Colab, execute this cell to clone the repo and set paths:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] Google Colab setup complete! Connected to PDE-Rover-Engine repository.')
except NameError:
    pass


# 🛞 Lab 2: Quasi-Linear First-Order PDEs, Lagrange's Method, and Rover Streamlines
### *Mission Control: Tracing Characteristic Trajectories Down the Potential Field*
**Course:** Applied Partial Differential Equations & Planetary Robotics
**Lecture Notes Reference:** Chapter 1, Sections 1.3.1 & 1.3.2 — **Notes pp. 11–16** *(PDF pp. 12–17)*

---
### 🗺️ Where You Are in the Rover Navigation Pipeline
```
[Lab 1: Terrain Slopes (p, q)] ──► 📍 [Lab 2: Streamline Paths (Lagrange)] ──► [Lab 3: Safety Buffers] ──► [Lab 4–5: Laplace Theory] ──► [Lab 6: Complete Rover Planner]
```

---
### 🧭 The Rosetta Stone: Connecting Lecture Notes to the Rover Engine

| 1. Lecture Notes Math (PDF pp. 12–17) | 2. Geometric & Physical Meaning | 3. Real Project Function (`rover_path_finder/`) |
| :--- | :--- | :--- |
| $P(x,y,z)p + Q(x,y,z)q = R(x,y,z)$ | **Quasi-linear 1st-order PDE:** linear in $(p, q)$ with direction coefficients $(P, Q, R)$ | Flow / transport equation along a vector field |
| $\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$ | **Lagrange's Auxiliary Equations:** ODEs governing the characteristic curves (streamlines) | Stepping `(x, y)` along the normalized gradient $(-gx, -gy)$ |
| $u(x,y,z) = c_1, \quad v(x,y,z) = c_2$ | Two independent first integrals whose intersection defines a trajectory curve | Streamline curve connecting the rover's `start` to `goal` |
| $\frac{dx}{-\partial\phi/\partial x} = \frac{dy}{-\partial\phi/\partial y}$ | **Gradient Descent Streamline:** the exact 2D path the rover drives down $\phi(x,y)$ | [`planners.descend(phi, hazard, start, goal)`](rover_path_finder/planners.py) |

---
## 1. Intuition First: Why Does Lagrange's Method Turn a PDE into a Trajectory Curve?

Recall from Calculus that a surface $z = f(x, y)$ can be written implicitly as $F(x, y, z) = f(x, y) - z = 0$, whose normal vector is $\mathbf{n} = (p, q, -1)$.
Now rewrite the quasi-linear PDE $P p + Q q = R$ as a dot product:
$$(P, Q, R) \cdot (p, q, -1) = 0$$

* Since the vector $(P, Q, R)$ is **perpendicular to the surface normal** $(p, q, -1)$ everywhere, the vector $(P, Q, R)$ must be **tangent to the surface**!
* Therefore, if the rover starts on the surface and moves with velocity $(dx/dt, dy/dt, dz/dt) = (P, Q, R)$, it traces out a curve—called a **characteristic curve (or streamline)**—given by **Lagrange's Auxiliary Equations**:
  $$\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$$
* In [`rover_path_finder/planners.py`](rover_path_finder/planners.py), once the navigation potential $\phi(x, y)$ is computed, the function [`descend()`](rover_path_finder/planners.py) steers the rover along the characteristic streamline of the velocity field $(P, Q) = (-\frac{\partial\phi}{\partial x}, -\frac{\partial\phi}{\partial y})$ from `start` to `goal`!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab2

%matplotlib inline
print("Lab 2 Environment Initialized! Ready for Streamline Trajectory Analysis.")


---
## 2. The Mathematical Recipe & All 3 Lagrange Solving Strategies (PDF pp. 12–17)

### Lagrange's Working Rule (Notes p. 11 / PDF p. 12):
To find the general solution of the quasi-linear PDE $P p + Q q = R$:
1. Write **Lagrange's Auxiliary Equations**:
   $$\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$$
2. Find **two independent first integrals** $u(x, y, z) = c_1$ and $v(x, y, z) = c_2$.
3. Write the general solution as $\Phi(u, v) = 0$ (or $u = \phi(v)$).

---
### 🛠️ How Do You Actually Solve $\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$? (The 3 Lecture Notes Strategies)

Students often ask: *"Once I write $\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$, how do I know which trick integrates it?"*
Your lecture notes (**Notes pp. 12–15 / PDF pp. 13–16**) teach **three distinct strategies** depending on how the variables $(x, y, z)$ are coupled:

---
#### 🔹 Strategy 1: Direct Pairing & Separation of Variables (Notes pp. 12–13, Examples 1 & 2 / PDF pp. 13–14)
* **When to use:** Two of the fractions contain **only those two variables** (the 3rd variable is missing or cancels out!).
* **📖 Worked Example 1 (Notes p. 12 / PDF p. 13):** Solve $2p - 3q = 2x$.
  1. Auxiliary equations: $\frac{dx}{2} = \frac{dy}{-3} = \frac{dz}{2x}$.
  2. **First pair ($\frac{dx}{2} = \frac{dy}{-3}$):** Contains only $dx$ and $dy$! Cross-multiply:
     $$-3\,dx = 2\,dy \implies 3x + 2y = c_1 = u$$
  3. **Second pair ($\frac{dx}{2} = \frac{dz}{2x}$):** Contains only $x$ and $z$! Multiply by $2$:
     $$x\,dx = dz \implies \frac{1}{2}x^2 - z = c_2 = v$$
  4. **General Solution:** $\Phi\left(3x + 2y,\; \frac{1}{2}x^2 - z\right) = 0$.

* **📖 Worked Example 2 (Notes pp. 12–13 / PDF p. 14):** Solve $x y p + y^2 q = z x y - 2x^2$.
  1. Auxiliary equations: $\frac{dx}{xy} = \frac{dy}{y^2} = \frac{dz}{zxy - 2x^2}$.
  2. **First pair ($\frac{dx}{xy} = \frac{dy}{y^2}$):** Cancel $y$ from both denominators to get $\frac{dx}{x} = \frac{dy}{y} \implies \ln|x| - \ln|y| = \ln|a| \implies \boxed{\frac{x}{y} = a}$.
  3. **Second pair:** Now use $y = \frac{x}{a}$ (or divide the 1st and 3rd denominators by $x$) to get $\frac{dx}{y} = \frac{dz}{zy - 2x}$, which reduces to a standard 1st-order linear ODE in $z(x)$!

---
#### 🔹 Strategy 2: Using the First Constant $a$ by Substitution to Unlock the Second Integral (Notes pp. 13–14, Example 3 / PDF pp. 14–15)
* **When to use:** One pair of fractions integrates easily to give $u(x,y,z) = a$, **but** the remaining fractions still have a pesky 3rd variable blocking integration!
* **How it works:** Temporarily replace the blocking variable using $u(x,y,z) = a$ (treating $a$ as a constant during integration!), integrate the second pair, and then **substitute the expression for $a$ back in at the very end**!
* **📖 Worked Example 3 (Notes pp. 13–14 / PDF pp. 14–15):** Solve $x^2 p + y^2 q = z^2$.
  1. **Auxiliary equations:**
     $$\frac{dx}{x^2} = \frac{dy}{y^2} = \frac{dz}{z^2}$$
  2. **First pair ($\frac{dx}{x^2} = \frac{dz}{z^2}$):** Integrate directly ($\int x^{-2}dx = -x^{-1}$):
     $$-\frac{1}{x} = -\frac{1}{z} + a' \implies \boxed{\frac{1}{x} - \frac{1}{z} = a} \qquad \text{(First Integral } u\text{)}$$
  3. **Second pair ($\frac{dx}{x^2} = \frac{dy}{y^2}$):** Integrate directly:
     $$\boxed{\frac{1}{x} - \frac{1}{y} = b} \qquad \text{(Second Integral } v\text{)}$$
  4. **How the PDF demonstrates the Substitution Trick (PDF p. 15):** Suppose your third denominator had both $x$ and $y$, e.g., if you needed to eliminate $x$ using $\frac{1}{x} = b + \frac{1}{y}$ before integrating, you treat $b$ as a constant while integrating, and once integrated, **replace $b$ back with $\left(\frac{1}{x} - \frac{1}{y}\right)$** so the final invariant contains only $(x, y, z)$!

---
#### 🔹 Strategy 3: The Equal-Fractions / Multiplier Rule (Notes pp. 14–15, Example 4 / PDF pp. 15–16)
* **When to use:** Every fraction has mixed variables $(x, y, z)$ in its denominator so **no two fractions separate directly**!
* **The Boxed Theorem (PDF p. 15):** By the algebra of equal ratios, for any multipliers $(\alpha, \beta, \gamma)$ (which can be numbers like $1, 1, 1$ or functions like $x, y, z$):
  $$\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R} = \frac{\alpha\,dx + \beta\,dy + \gamma\,dz}{\alpha P + \beta Q + \gamma R}$$
  > 🎯 **Key Insight:** If you choose multipliers $(\alpha, \beta, \gamma)$ so that the **denominator sum equals ZERO**:
  > $$\alpha P + \beta Q + \gamma R = 0 \implies \boxed{\alpha\,dx + \beta\,dy + \gamma\,dz = 0}$$
  > Then the numerator must also equal **zero**, which you can integrate immediately!

* **📖 Worked Example 4 (Notes pp. 14–15 / PDF pp. 15–16):** Solve:
  $$(y - z)p + (z - x)q = x - y$$
  1. **Auxiliary equations:**
     $$\frac{dx}{y - z} = \frac{dy}{z - x} = \frac{dz}{x - y}$$
     *(Notice that $x, y, z$ are tangled in all three denominators—direct pairing fails!)*
  2. **First Set of Multipliers $(\alpha, \beta, \gamma) = (1, 1, 1)$:**
     Add the denominators directly:
     $$(y - z) + (z - x) + (x - y) = 0$$
     Since the denominator sum is $0$, the numerator sum must also be $0$:
     $$1\cdot dx + 1\cdot dy + 1\cdot dz = 0 \implies \int dx + \int dy + \int dz = a \implies \boxed{x + y + z = a} \qquad \text{(First Integral } u\text{)}$$
  3. **Second Set of Multipliers $(\alpha, \beta, \gamma) = (x, y, z)$:**
     Multiply the top and bottom of the three fractions by $x$, $y$, and $z$ respectively and add:
     $$\text{Denominator: } x(y - z) + y(z - x) + z(x - y) = xy - xz + yz - yx + zx - zy = 0$$
     Again the denominator sum is $0$, so the numerator must vanish:
     $$x\,dx + y\,dy + z\,dz = 0 \implies \frac{1}{2}x^2 + \frac{1}{2}y^2 + \frac{1}{2}z^2 = \frac{b}{2} \implies \boxed{x^2 + y^2 + z^2 = b} \qquad \text{(Second Integral } v\text{)}$$
  4. **General Solution:** $\Phi(x + y + z,\; x^2 + y^2 + z^2) = 0$! *(Geometrically, the characteristic curves are the intersections of planes $x+y+z=a$ with spheres $x^2+y^2+z^2=b$, i.e., circles tilted in 3D space!)*

---
### ⚠️ Crucial Integration Tip: Logarithms & Ratios (`ln|x| - ln|y| = ln|c|`)
When integrating equations like $\frac{dx}{x} = \frac{dy}{y}$ (such as **Exercise 5 on Notes p. 16 / PDF p. 17**):
$$\int \frac{dx}{x} = \int \frac{dy}{y} \implies \ln|x| - \ln|y| = \ln|c_1| \implies \ln\left|\frac{x}{y}\right| = \ln|c_1| \implies \frac{x}{y} = c_1$$
Writing the constant of integration as $\ln|c_1|$ lets you combine the logarithms cleanly into a simple algebraic ratio $u = \frac{x}{y}$!


---
## 3. 📝 Paper & Pen Challenge 1 (Notes p. 16, Exercise 5 / PDF p. 17)

**Problem:** Find two independent invariants $u(x, y) = c_1$ and $v(y, z) = c_2$ for Euler's homogeneous PDE:
$$x p + y q = z$$

**Work through these steps on paper:**
1. Set up Lagrange's auxiliary equations: $\frac{dx}{x} = \frac{dy}{y} = \frac{dz}{z}$.
2. From the first pair $\frac{dx}{x} = \frac{dy}{y}$, integrate both sides to obtain the ratio $u(x, y) = c_1$.
3. From the second pair $\frac{dy}{y} = \frac{dz}{z}$, integrate both sides to obtain the ratio $v(y, z) = c_2$.

<details>
<summary>💡 <b>Hint (Combining Logarithms)</b></summary>

Since $\ln|x| - \ln|y| = \ln|x/y| = \text{const}$, the first invariant is `"x/y"` (or `"y/x"`), and similarly from $\frac{dy}{y} = \frac{dz}{z}$ the second invariant is `"y/z"` (or `"z/y"`).
</details>


In [ ]:
# GRADED CELL: Paper Challenge 1
### START CODE HERE ###
student_u = "x/y"
student_v = "y/z"
### END CODE HERE ###

# Run numerical verification test:
test_lab2.check_paper_challenge_1(student_u, student_v)


---
## 4. 🚀 Rover Bridge Challenge (Analytical Streamline Trajectory)

Suppose in a local valley near the goal $(0, 0)$, the navigation potential field is approximated by the smooth bowl:
$$\phi(x, y) = x^2 + 2y^2$$
The rover drives in the direction of steepest descent $\mathbf{v} = -\nabla\phi = \left(-\frac{\partial\phi}{\partial x}, -\frac{\partial\phi}{\partial y}\right) = (-2x, -4y)$, so any quantity $z$ carried along the streamline satisfies:
$$-2x\,p - 4y\,q = 0$$

**On your paper:**
1. Write the first pair of Lagrange's auxiliary equations: $\frac{dx}{-2x} = \frac{dy}{-4y}$.
2. Divide both denominators by $-2x y$ to separate variables: $\frac{2\,dx}{x} = \frac{dy}{y}$.
3. Integrate both sides ($2\ln|x| + \ln|k| = \ln|y|$) and exponentiate to express $y$ as a function of $x$ and arbitrary constant $k$.

<details>
<summary>💡 <b>Hint 1 (Integrating Separated Variables)</b></summary>

Integrating $\frac{dy}{y} = 2\frac{dx}{x}$ gives $\ln|y| = 2\ln|x| + C = \ln(x^2) + \ln|k| = \ln|k x^2|$.
</details>

<details>
<summary>💡 <b>Hint 2 (Removing the Logarithm)</b></summary>

Exponentiating both sides yields the parabolic family $y = k x^2$.
</details>


In [ ]:
# GRADED CELL: Rover Streamline Challenge
### START CODE HERE ###
rover_streamline = "y = k*x^2"
### END CODE HERE ###

# Run numerical verification test:
test_lab2.check_paper_challenge_2(rover_streamline)


---
## 5. 💻 Python Implementation: Numerical Characteristic Streamline Tracer

On a real Martian terrain grid, the potential $\phi(x, y)$ is stored as a 2D array rather than a simple formula. To trace Lagrange's characteristic equations $\frac{dx}{dt} = P(x, y),\; \frac{dy}{dt} = Q(x, y)$ numerically—just like [`planners.descend()`](rover_path_finder/planners.py) in our rover engine—we step forward iteratively from `start`:
$$\mathbf{x}_{k+1} = \mathbf{x}_k + \Delta t \cdot \mathbf{v}(x_k, y_k)$$

Complete `trace_streamline_rk4` below to trace the rover's path over `steps` time steps.


In [ ]:
# GRADED FUNCTION: trace_streamline_rk4
def trace_streamline_rk4(velocity_fn, start=(1.0, 2.0), dt=0.01, steps=50):
    # Numerically traces a characteristic curve (streamline) of a quasi-linear system
    trajectory = [start]
    curr = np.array(start, dtype=float)

    ### START CODE HERE ###
    for _ in range(steps):
        vx, vy = velocity_fn(curr[0], curr[1])
        v = np.array([vx, vy], dtype=float)
        curr = curr + dt * v
        trajectory.append(tuple(curr))
    ### END CODE HERE ###

    return np.array(trajectory)

# Run automated unit test:
test_lab2.test_rk4_streamline_tracer(trace_streamline_rk4)


---
## 6. 🎨 Visualizing the Physics: Exact Paper Solution vs. Numerical Streamline Tracer

Starting from $(x_0, y_0) = (1, 2)$, our paper solution $y = k x^2$ gives $2 = k(1)^2 \implies k = 2$, so the exact Lagrange characteristic curve is $y = 2x^2$.
Run the cell below to verify that your numerical tracer rides the exact same curve $y = 2x^2$ down to the goal $(0, 0)$!


In [ ]:
def sample_velocity(x, y):
    return -2.0 * x, -4.0 * y

traj = trace_streamline_rk4(sample_velocity, start=(1.0, 2.0), dt=0.02, steps=120)

x_vals = np.linspace(-0.2, 1.2, 25)
y_vals = np.linspace(-0.2, 2.2, 25)
X, Y = np.meshgrid(x_vals, y_vals)
PHI = X**2 + 2 * Y**2
U, V = sample_velocity(X, Y)

x_exact = np.linspace(0.05, 1.0, 100)
y_exact = 2.0 * (x_exact ** 2)

plt.figure(figsize=(8.5, 6))
plt.contour(X, Y, PHI, levels=12, cmap='viridis', alpha=0.5)
plt.streamplot(X, Y, U, V, color='lightgray', density=0.9)
plt.plot(x_exact, y_exact, 'b--', linewidth=3.5, label=r"Paper Solution (Lagrange): $y = 2x^2$")
plt.plot(traj[:, 0], traj[:, 1], 'r-', linewidth=2.2, label="Numerical Tracer (Code)")
plt.plot(1.0, 2.0, 'go', markersize=9, label="Start (1, 2)")
plt.plot(0.0, 0.0, 'y*', markersize=13, label="Goal (0, 0)")
plt.title("Lagrange Characteristic Curve vs. Numerical Streamline Tracer", fontsize=12)
plt.xlabel("x")
plt.ylabel("y")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.show()

print("Lab 2 Takeaways:")
print("  1) Quasi-linear PDEs P*p + Q*q = R are solved via Lagrange's auxiliary ODEs dx/P = dy/Q = dz/R.")
print("  2) Lagrange's characteristic curves are the exact physical streamlines of the rover.")
print("  3) planners.descend() in our codebase is a discrete Lagrange characteristic tracer!")
